# 09. Lecke - Metakogníció Tervezési Minta


## Beállítás

Ez a jegyzetfüzet a Metakogníció tervezési mintát mutatja be a Microsoft Agent Framework segítségével.

**Előfeltételek:**
- Azure OpenAI telepítés környezeti változók használatával konfigurálva
- Azure CLI hitelesítve (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mi az a metakogníció?

A metakogníció a **gondolkodás a gondolkodásról**. Az MI-ügynökök esetében ez azt jelenti, hogy olyan ügynököket építünk, amelyek képesek:

- **Önreflexiót gyakorolni** saját eredményeikre és gondolkodási folyamatukra
- **Hibákat felismerni** és elegánsan helyreállni, ahelyett, hogy csendben hibáznának
- **Értékelni**, hogy válaszaik teljesek és hasznosak-e
- **Alkalmazkodni** stratégiájában, ha az első megközelítés nem működik (pl. tartalék rendszerre váltás)

Egy metakognitív ügynök nemcsak kérdésekre válaszol — figyelemmel kíséri saját teljesítményét és menet közben igazít.


## Elsődleges és tartalék eszközök

Egy gyakori metakogníciós minta az **alternatív stratégia**. Az ügynök először egy elsődleges eszközt próbál meg; ha az kudarcot vall (például 404-es hiba), az ügynök felismeri a hibát és átlátható módon átvált egy tartalék eszközre.

Ez visszatükrözi a való életbeli rendszereket, ahol az elsődleges szolgáltatások esetleg nem elérhetők, és az ügynököknek saját maguknak kell diagnosztizálniuk a problémát, mielőtt alternatív útvonalat választanak.

Lent két járatkereső eszközt definiálunk:
- **Elsődleges** — Párizsra, Tokióra és Barcelonára vonatkozik
- **Tartalék** — Berlinre, Sydneyre és New York City-re vonatkozik


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Önkritikus ügynök hibajavítással

Az alábbi ügynököt úgy utasították, hogy először próbálja meg a fő repülési rendszert használni, ismerje fel a hibákat, és áttetsző módon térjen vissza a tartalék rendszerre. Minden válasz után röviden önértékeli, hogy teljesen megválaszolta-e a felhasználó kérdését.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Önértékelési minta

A metakogníció egy másik aspektusa az **önértékelés**: egy külön ügynök (vagy ugyanaz az ügynök egy második körben) átnézi a választ teljesség, pontosság és hasznosság szempontjából.

Lent létrehozunk egy `ResponseEvaluator` ügynököt, amely három dimenzióban értékeli az utazási ügynöki válaszokat.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Összefoglaló

Ebben a leckében megtanultad, hogyan építsünk **metakognitív ügynököket** a Microsoft Agent Framework használatával:

- **Önreflexió**: Ügynökök, amelyek figyelik saját gondolkodásukat, és átláthatóan kommunikálják, mi történt.
- **Hibakezelés tartalékkal**: Elsődleges és biztonsági eszköz mintája, ahol az ügynök hibákat észlel (pl. 404 hibák), és automatikusan megpróbál egy alternatív forrást.
- **Önértékelés**: Egy különértékelő ügynök, amely pontozza a válaszokat teljesség, pontosság és hasznosság alapján.

Ezek a minták robosztusabbá, átláthatóbbá és megbízhatóbbá teszik az ügynököket — létfontosságú tulajdonságok a gyártási környezetben való alkalmazáshoz.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
